In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
from sklearn.metrics.pairwise import cosine_similarity


OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "bge-m3"
LLM_MODEL = "llama3.2"

EMBEDDINGS_FILE = "embeddings.joblib"
TOP_RESULTS = 5
TIMEOUT = 180


# ==========================================
#  EMBEDDINGS
# ==========================================

def create_embedding(text_list):
    url = f"{OLLAMA_URL}/api/embed"

    response = requests.post(
        url,
        json={
            "model": EMBEDDING_MODEL,
            "input": text_list
        },
        timeout=TIMEOUT
    )

    response.raise_for_status()
    data = response.json()

    if "embeddings" not in data:
        raise ValueError(f"Embedding error: {data}")

    return data["embeddings"]


# ==========================================
# GENERATING  AI RESPONSE
# ==========================================

def inference(prompt):
    url = f"{OLLAMA_URL}/api/generate"

    response = requests.post(
        url,
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=TIMEOUT
    )

    response.raise_for_status()
    data = response.json()

    if "response" not in data:
        raise ValueError(f"Generation error: {data}")

    # Return only the answer, not the complete JSON.
    return data["response"].strip()


# ==========================================
# LOADING EMBEDDINGS
# ==========================================

try:
    df = joblib.load(EMBEDDINGS_FILE)

except FileNotFoundError:
    raise SystemExit(
        f"ERROR: {EMBEDDINGS_FILE} not found. "
        "Keep it in the same folder as this script."
    )

if not isinstance(df, pd.DataFrame):
    raise SystemExit(
        "ERROR: embeddings.joblib must contain a DataFrame."
    )

required_columns = [
    "title",
    "number",
    "start",
    "end",
    "text",
    "embedding"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise SystemExit(
        f"Missing columns in embeddings.joblib: {missing_columns}"
    )

df = df.dropna(
    subset=["text", "embedding"]
).copy()

if df.empty:
    raise SystemExit("ERROR: No valid chunks found.")

df["text"] = df["text"].astype(str)

try:
    embedding_matrix = np.vstack(
        df["embedding"].apply(
            lambda value: np.asarray(value, dtype=np.float32)
        ).values
    )

except (ValueError, TypeError) as error:
    raise SystemExit(
        f"Invalid stored embeddings: {error}"
    )

if embedding_matrix.ndim != 2:
    raise SystemExit("ERROR: Invalid embedding matrix.")

if not np.isfinite(embedding_matrix).all():
    raise SystemExit("ERROR: Embeddings contain invalid values.")


# ==========================================
# RAG QUESTION ANSWERING
# ==========================================

print("\n======================================")
print(" SIGMA WEB DEVELOPMENT AI ASSISTANT")
print("======================================")
print("Ask questions about your course.")
print("Type 'exit' to stop.\n")


while True:

    incoming_query = input("Ask a Question: ").strip()

    if incoming_query.lower() in ["exit", "quit"]:
        print("Assistant stopped.")
        break

    if not incoming_query:
        print("Please enter a question.\n")
        continue

    try:

        # Step 1: Create query embedding
        question_embedding = np.asarray(
            create_embedding([incoming_query])[0],
            dtype=np.float32
        )

        if question_embedding.ndim != 1:
            raise ValueError("Invalid question embedding.")

        if not np.isfinite(question_embedding).all():
            raise ValueError(
                "Question embedding contains invalid values."
            )

        # Step 2: Check dimensions
        if embedding_matrix.shape[1] != len(question_embedding):
            raise ValueError(
                "Embedding dimension mismatch. "
                "Recreate embeddings using the same model: "
                f"{EMBEDDING_MODEL}"
            )

        # Step 3: Calculate similarity
        similarities = cosine_similarity(
            embedding_matrix,
            question_embedding.reshape(1, -1)
        ).flatten()

        # Step 4: Retrieve most relevant chunks
        top_k = min(TOP_RESULTS, len(df))

        top_indices = np.argsort(similarities)[::-1][:top_k]

        new_df = df.iloc[top_indices].copy()

        new_df["similarity"] = similarities[top_indices]

        # Step 5: Prepare course context
        context_columns = [
            "title",
            "number",
            "start",
            "end",
            "text"
        ]

        context = new_df[context_columns].to_json(
            orient="records",
            force_ascii=False
        )

        # Step 6: Create prompt
        prompt = f"""
You are an AI Teaching Assistant for the
Sigma Web Development course.

Use the following retrieved course subtitle chunks
to answer the student's question.

COURSE MATERIAL:
{context}

STUDENT QUESTION:
{incoming_query}

STRICT INSTRUCTIONS:

1. Answer clearly, naturally, and concisely.
2. Use the supplied course material as your evidence.
3. Identify the relevant video title and video number.
4. Give timestamps using the supplied start and end values.
5. Explain briefly what is taught at those timestamps.
6. If multiple chunks refer to the same video and topic,
   combine them into one useful explanation.
7. Do not repeat the same information.
8. Do not invent video titles, numbers, timestamps, or facts.
9. If the retrieved material is insufficient, say so honestly.
10. If the question is unrelated to web development course
    content, politely explain that you only answer questions
    related to this course.
11. Do not claim that a topic is taught at a particular time
    unless the supplied material supports that claim.
12. Make the answer easy for a student to understand.

FORMAT:

Answer:
Give the direct answer.

Where to learn:
Video number and title, if available.

Timestamp:
Start time and end time in seconds.

What you will learn:
Briefly explain the relevant content.

Only include details supported by the course material.
Do not display the raw JSON or internal processing information.
"""

        # Save prompt for debugging
        with open(
            "prompt.txt",
            "w",
            encoding="utf-8"
        ) as file:
            file.write(prompt)

        # Step 7: Generate answer
        answer = inference(prompt)

        # Step 8: Print only the final answer
        print("\n" + "=" * 45)
        print("AI TEACHING ASSISTANT")
        print("=" * 45)
        print(answer)
        print("=" * 45 + "\n")

        # Step 9: Save only the answer
        with open(
            "response.txt",
            "w",
            encoding="utf-8"
        ) as file:
            file.write(answer)

    except requests.exceptions.ConnectionError:
        print(
            "\nERROR: Cannot connect to Ollama. "
            "Make sure Ollama is running.\n"
        )

    except requests.exceptions.Timeout:
        print(
            "\nERROR: Ollama took too long to respond. "
            "Try again.\n"
        )

    except requests.exceptions.RequestException as error:
        print(f"\nOllama API ERROR: {error}\n")

    except Exception as error:
        print(f"\nERROR: {error}\n")


 SIGMA WEB DEVELOPMENT AI ASSISTANT
Ask questions about your course.
Type 'exit' to stop.



Ask a Question:  where is video tag taught in the course



AI TEACHING ASSISTANT
The video tag is taught in the course in "Video, Audio & Media in HTML" at video number 10, from 140.28 to 141.92 seconds, and also from 527.08 to 528.08 seconds.



Ask a Question:  what is flexbox in css



AI TEACHING ASSISTANT
The concept of Flexbox in CSS is explained in the course material.

To learn about Flexbox in CSS, watch Video 8, "Inline & Block Elements in HTML".

Timestamp: 501.0 seconds to 505.0 seconds.

What you will learn: Flexbox, inline flex, inline block, and more on HTML, Flexbox Grid etc.



Ask a Question:  explain html in simple words



AI TEACHING ASSISTANT
HTML stands for HyperText Markup Language, which is the standard markup language used to create web pages.

At video 1, we have a brief explanation of HTML (timestamps: 500.9 - 501.9). The basic code of HTML is discussed in this section.

 HTML introduction covers basic HTML elements such as headings, paragraphs, and links (video 4, timestamps: 367.4 - 368.6).

We also learn about the different types of CSS (Cascading Style Sheets) in relation to HTML, but the explanation of HTML itself is not covered in detail.

Let me know if you'd like me to help with anything else.



Ask a Question:  what is css



AI TEACHING ASSISTANT
Answer:
CSS stands for Cascading Style Sheets.

Where to learn:
Video 1: "Installing VS Code & How Websites Work" (start: 1121.0, end: 1124.0)

What you will learn: CSS is used to style a page, explaining its role in beautifying HTML content.

